# Weather Trend Forecasting — Global Weather Repository

> **PM Accelerator mission.** Product Manager Accelerator (PMA), led by Dr. Nancy Li, supports PM professionals at every career stage through hands-on AI product training, coaching, and a sharing-first community; its PMA Kids nonprofit brings free PM education to underserved teenagers. Source: https://www.pmaccelerator.io/

This notebook is a **narrative over the `weather_forecast` package** — all logic lives in `src/`, nothing is redefined here. It runs top-to-bottom on a fresh kernel (Restart & Run All), using the real Kaggle CSV if present in `data/raw/`, else a deterministic synthetic fallback.

In [ ]:
import json
import sys
import warnings

warnings.filterwarnings("ignore")
import pandas as pd
from IPython.display import Image, Markdown, display

from weather_forecast import cleaning, config, ingest, pipeline, profiling
from weather_forecast import engineering as fe

pipeline.set_global_seeds()
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass

print(config.mission_banner())

# Self-contained: regenerate all artifacts once if they are not present yet.
if not (config.METRICS_DIR / "model_comparison.csv").exists():
    print("\nArtifacts missing -> running the full pipeline once (this takes ~1 min)...")
    pipeline.run_all()

FIG, MET, PROC = config.FIGURES_DIR, config.METRICS_DIR, config.PROCESSED_DIR

def show_fig(name, caption=""):
    p = FIG / name
    if p.exists():
        display(Image(filename=str(p)))
        if caption:
            display(Markdown(f"*{caption}*"))
    else:
        display(Markdown(f"_(figure {name} not found)_"))

def show_csv(name, n=10):
    p = MET / name
    return pd.read_csv(p).head(n) if p.exists() else f"(missing {name})"


## 1. Data loading & cleaning
tz-aware UTC parsing, imperial twins dropped, dedup, physical gating, per-city imputation, outlier flagging — see `ingest.py` / `cleaning.py`.

In [ ]:
df, is_real = ingest.load_weather()
print("data source:", "REAL Kaggle CSV" if is_real else "SYNTHETIC fallback", "| raw shape:", df.shape)
clean = cleaning.clean(df)
print("clean shape:", clean.shape,
      "| grain unique:", not clean.duplicated(config.KEY_COLS + [config.TIME_COL]).any())
clean[[config.TIME_COL, "country", "location_name", config.TARGET, "humidity", "air_quality_pm2_5"]].head()


## 2. Exploratory analysis
Multimodal global temperature, **hemisphere anti-phase** seasonality, zero-inflated precipitation, Spearman correlations (rank-robust).

In [ ]:
display(Markdown("**Grain report**")); display(profiling.grain_report(clean))
for fn in ["eda_temperature_distribution.png", "eda_seasonal_cycle.png",
           "eda_precipitation_analysis.png", "eda_correlation_spearman.png",
           "eda_temperature_trend.png", "eda_global_aggregate.png"]:
    show_fig(fn)


## 3. Anomaly detection (advanced EDA)
Per-(city, season) robust z-scores + STL + IsolationForest ∩ LOF, then *analysis* (condition lift, ingestion-artifact detection).

In [ ]:
an = pd.read_parquet(PROC / "anomalies.parquet")
print("anomalies flagged:", len(an), "| high-confidence:", int(an["is_high_confidence"].sum()))
show_fig("anomaly_geography.png", "Anomalies are defined relative to (city, month) - never the global pool.")
display(Markdown("**Anomaly lift by weather condition** (sanity check: real weather, not noise)"))
display(show_csv("anomaly_condition_lift.csv", 8))


## 4. Feature engineering & leakage guards
Calendar/cyclical + per-city lags & shifted rollings; exogenous weather only lagged. The asserts below are the core correctness story.

In [ ]:
feats = fe.build_features(clean)
fcols = fe.feature_columns(feats)
print("n model features:", len(fcols))
print("examples:", fcols[:12])

# Leakage guards demonstrated live:
g = feats[feats.location_name == feats.location_name.iloc[0]].sort_values(config.TIME_COL)
lag_ok = (g[config.TARGET].shift(1).dropna().round(3).values
          == g[f"{config.TARGET}_lag1"].dropna().round(3).values).all()
print("lag1 == prior-row target (causal):", bool(lag_ok))
print("contemporaneous 'humidity' excluded:", "humidity" not in fcols,
      "| only 'humidity_lag1' used:", "humidity_lag1" in fcols)
print("leaky 'feels_like_*' excluded:", not any("feels_like" in c for c in fcols))


## 5. Backtesting & model comparison
Leakage-safe rolling-origin CV on a global date axis. **MASE** is the primary metric; baselines are mandatory and the ensemble drops anything worse than seasonal-naive.

In [ ]:
display(Markdown("**Model comparison (representative cities), sorted by MASE** - MASE<1 beats seasonal-naive"))
display(show_csv("model_comparison.csv"))
show_fig("forecast_model_comparison.png", "SARIMAX and the global GBM tie on representative cities; the GBM wins on the full panel.")
print("ensemble weights:", json.loads((MET / "ensemble_weights.json").read_text()))
print("stacking weights:", json.loads((MET / "stacking_weights.json").read_text()))
display(Markdown("**7-day-ahead temperature forecast per representative city**"))
display(pd.read_parquet(PROC / "predictions.parquet").head(14))


## 6. Unique analyses
Climate (latitude zones), environmental impact (air quality ↔ weather), feature importance, and spatial / cross-country geography.

In [ ]:
for fn in ["climate_seasonal_cycle.png", "climate_zone_distributions.png",
           "air_quality_weather_heatmap.png", "air_quality_epa_by_continent.png",
           "feature_importance_permutation.png", "spatial_temperature_map.png",
           "spatial_country_choropleth.png", "spatial_continent_box.png"]:
    show_fig(fn)
display(Markdown("**Top permutation feature importances** (validation-only, robust)"))
display(show_csv("feature_importance_permutation.csv", 10))


## 7. Conclusions
- On representative cities it is a **statistical tie** between SARIMAX (MASE ≈ 0.79) and the global gradient-boosting model (XGBoost ≈ 0.79); the **global model wins on the full 268-city panel (MASE ≈ 0.73)** and is the deployable choice — one model, generalizes to new cities, no per-city refit.
- **Leakage is controlled** by construction (chronological CV, shifted per-city features, lagged exogenous, target-twin exclusion) and verified by tests.
- Short snapshot history limits annual seasonality / trend claims — framed honestly throughout.

Full write-up: [`REPORT.md`](../REPORT.md) · methodology: [`docs/TECHNICAL_SPEC.md`](../docs/TECHNICAL_SPEC.md).